# SOLUSDT order hedging check

Assumptions for hedging success:
- Match open and hedge orders by `strategy_id`.
- Required hedge qty = open `filled_qty` (not the original order qty).
- Hedged ok when sum of hedge `filled_qty` >= required qty (within a small tolerance).
- Rows with open `filled_qty == 0` are excluded from the unhedged list.


In [1]:
import pandas as pd

engine_used = "pyarrow"
try:
    df = pd.read_parquet("WLDUSDT_order.parquet", engine="pyarrow")
except Exception:
    engine_used = "fastparquet"
    df = pd.read_parquet("WLDUSDT_order.parquet", engine="fastparquet")

print(f"engine={engine_used}, rows={len(df)}, cols={df.shape[1]}")

df.head()


engine=pyarrow, rows=6287, cols=22


,symbol,order_id,client_order_id,side,order_type,price,quantity,status,trading_venue,strategy_id,...,order_side,create_ts,opening_bid0,opening_ask0,hedging_bid0,hedging_ask0,price_offset,update_ts,local_ts,filled_qty
0,WLD-USDT-SWAP,3152205949550190599,2452416932922523649,BUY,LIMIT,0.5030,397.0,CANCELED,OkexFutures,570997813,...,open,1.766445e+15,0.5038,0.5039,0.5041,0.5042,0.0015,1.766446e+18,1766445457940000000,0.0
1,WLD-USDT-SWAP,3152205949550190598,2452415292245016577,BUY,LIMIT,0.4992,400.0,CANCELED,OkexFutures,570997431,...,open,1.766445e+15,0.5038,0.5039,0.5041,0.5042,0.0090,1.766446e+18,1766445457940000000,0.0
2,WLD-USDT-SWAP,3152205949550190597,2452413372394635265,BUY,LIMIT,0.5035,397.0,CANCELED,OkexFutures,570996984,...,open,1.766445e+15,0.5038,0.5039,0.5041,0.5042,0.0005,1.766446e+18,1766445457940000000,0.0
3,WLD-USDT-SWAP,3152205949550190596,2452417761851211777,BUY,LIMIT,0.5027,397.0,CANCELED,OkexFutures,570998006,...,open,1.766445e+15,0.5038,0.5039,0.5041,0.5042,0.0020,1.766446e+18,1766445457940000000,0.0
4,WLD-USDT-SWAP,3152205949550190595,2452416112583770113,BUY,LIMIT,0.5031,397.0,CANCELED,OkexFutures,570997622,...,open,1.766445e+15,0.5038,0.5039,0.5041,0.5042,0.0012,1.766446e+18,1766445457940000000,0.0


In [2]:
open_df = df[df["order_side"] == "open"].copy()
hedge_df = df[df["order_side"] == "hedge"].copy()

open_df = open_df.rename(
    columns={
        "order_id": "open_order_id",
        "client_order_id": "open_client_order_id",
        "status": "open_status",
        "side": "open_side",
        "quantity": "open_qty",
        "filled_qty": "open_filled_qty",
        "create_ts": "open_create_ts",
        "update_ts": "open_update_ts",
        "local_ts": "open_local_ts",
    }
)

open_df["open_local_ts_utc"] = pd.to_datetime(
    open_df["open_local_ts"], unit="ns", utc=True
)


open_cols = [
    "symbol",
    "open_order_id",
    "open_client_order_id",
    "open_side",
    "order_type",
    "price",
    "open_qty",
    "open_status",
    "trading_venue",
    "strategy_id",
    "opening_venue",
    "hedging_venue",
    "open_create_ts",
    "open_update_ts",
    "open_local_ts",
    "open_local_ts_utc",
    "open_filled_qty",
]

open_df = open_df[open_cols].set_index("strategy_id")

hedge_agg = hedge_df.groupby("strategy_id").agg(
    hedge_order_count=("order_id", "size"),
    hedge_filled_count=("status", lambda s: (s == "FILLED").sum()),
    hedge_canceled_count=("status", lambda s: (s == "CANCELED").sum()),
    hedge_filled_qty_sum=("filled_qty", "sum"),
)

summary = open_df.join(hedge_agg, how="left")

for col in ["hedge_order_count", "hedge_filled_count", "hedge_canceled_count"]:
    summary[col] = summary[col].fillna(0).astype(int)
summary["hedge_filled_qty_sum"] = summary["hedge_filled_qty_sum"].fillna(0.0)

summary.head()


,symbol,open_order_id,open_client_order_id,open_side,order_type,price,open_qty,open_status,trading_venue,opening_venue,hedging_venue,open_create_ts,open_update_ts,open_local_ts,open_filled_qty,hedge_order_count,hedge_filled_count,hedge_canceled_count,hedge_filled_qty_sum
strategy_id,,,,,,,,,,,,,,,,,,,
570997813,WLD-USDT-SWAP,3152205949550190599,2452416932922523649,BUY,LIMIT,0.5030,397.0,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766445e+15,1.766446e+18,1766445457940000000,0.0,0,0,0,0.0
570997431,WLD-USDT-SWAP,3152205949550190598,2452415292245016577,BUY,LIMIT,0.4992,400.0,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766445e+15,1.766446e+18,1766445457940000000,0.0,0,0,0,0.0
570996984,WLD-USDT-SWAP,3152205949550190597,2452413372394635265,BUY,LIMIT,0.5035,397.0,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766445e+15,1.766446e+18,1766445457940000000,0.0,0,0,0,0.0
570998006,WLD-USDT-SWAP,3152205949550190596,2452417761851211777,BUY,LIMIT,0.5027,397.0,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766445e+15,1.766446e+18,1766445457940000000,0.0,0,0,0,0.0
570997622,WLD-USDT-SWAP,3152205949550190595,2452416112583770113,BUY,LIMIT,0.5031,397.0,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766445e+15,1.766446e+18,1766445457940000000,0.0,0,0,0,0.0


In [3]:
summary["need_hedge_qty"] = summary["open_filled_qty"]

tol = 1e-9
summary["hedged_ok"] = summary["hedge_filled_qty_sum"] + tol >= summary["need_hedge_qty"]

need_hedge = summary[summary["need_hedge_qty"] > 0]
unhedged = need_hedge[~need_hedge["hedged_ok"]].sort_values("open_create_ts")

print(f"open rows: {len(open_df)}")
print(f"need hedge rows: {len(need_hedge)}")
print(f"unhedged rows: {len(unhedged)}")

unhedged


open rows: 3767
need hedge rows: 957
unhedged rows: 11


,symbol,open_order_id,open_client_order_id,open_side,order_type,price,open_qty,open_status,trading_venue,opening_venue,...,open_create_ts,open_update_ts,open_local_ts,open_filled_qty,hedge_order_count,hedge_filled_count,hedge_canceled_count,hedge_filled_qty_sum,need_hedge_qty,hedged_ok
strategy_id,,,,,,,,,,,,,,,,,,,,,
332253885,WLD-USDT-SWAP,3147514195123101696,1427019570043944961,SELL,LIMIT,0.5179,386.0,FILLED,OkexFutures,OkexFutures,...,1.766306e+15,1.766306e+18,1766305632762000000,386.0,6,1,5,121.0,386.0,False
2013442052,WLD-USDT-SWAP,3147570606431526912,8647667765731131393,SELL,LIMIT,0.5196,384.0,FILLED,OkexFutures,OkexFutures,...,1.766307e+15,1.766307e+18,1766307313950000000,384.0,1,0,0,0.0,384.0,False
2079253536,WLD-USDT-SWAP,3148869851419762688,8930325937212358657,SELL,LIMIT,0.5031,397.0,FILLED,OkexFutures,OkexFutures,...,1.766346e+15,1.766346e+18,1766346034468000000,397.0,5,1,4,302.0,397.0,False
947272611,WLD-USDT-SWAP,3149048041090048000,4068504884641529857,SELL,LIMIT,0.5036,397.0,CANCELED,OkexFutures,OkexFutures,...,1.766351e+15,1.766351e+18,1766351344934000000,4.0,0,0,0,0.0,4.0,False
1787473803,WLD-USDT-SWAP,3149580636898975744,7677141526341746689,SELL,LIMIT,0.5179,386.0,FILLED,OkexFutures,OkexFutures,...,1.766367e+15,1.766367e+18,1766367217526000000,386.0,1,1,0,331.0,386.0,False
1781391640,WLD-USDT-SWAP,3149652490426769408,7651018835167805441,BUY,LIMIT,0.5087,393.0,FILLED,OkexFutures,OkexFutures,...,1.766369e+15,1.766369e+18,1766369358928000000,393.0,1,1,0,235.0,393.0,False
1781391882,WLD-USDT-SWAP,3149652490258997248,7651019874549891073,BUY,LIMIT,0.5086,393.0,FILLED,OkexFutures,OkexFutures,...,1.766369e+15,1.766369e+18,1766369358923000000,393.0,1,1,0,189.0,393.0,False
1166398612,WLD-USDT-SWAP,3149920084874485760,5009643892639793153,SELL,LIMIT,0.5166,387.0,CANCELED,OkexFutures,OkexFutures,...,1.766377e+15,1.766377e+18,1766377333864000000,6.0,0,0,0,0.0,6.0,False
1951414857,WLD-USDT-SWAP,3150594943992455171,8381262991743516673,BUY,LIMIT,0.5154,388.0,CANCELED,OkexFutures,OkexFutures,...,1.766397e+15,1.766398e+18,1766397446233000000,2.0,0,0,0,0.0,2.0,False


In [4]:
def show_strategy(strategy_id: int):
    return df[df["strategy_id"] == strategy_id].sort_values(
        ["order_side", "create_ts", "order_id"]
    )

# Example: show_strategy(1870436056)
